In [6]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:
df = pd.read_csv("/content/job_recommendation_dataset.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 50000
Columns: 7


In [8]:
df.head()

,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills
0,Early years teacher,Richardson Ltd,Sydney,Senior Level,87000.0,Healthcare,Pharmaceuticals
1,Counselling psychologist,"Ramos, Santiago and Stewart",San Francisco,Mid Level,50000.0,Marketing,"Google Ads, SEO, Content Writing"
2,Radio broadcast assistant,Franco Group,New York,Mid Level,77000.0,Healthcare,"Patient Care, Nursing, Medical Research, Pharm..."
3,"Designer, exhibition/display",Collins Inc,Berlin,Senior Level,90000.0,Software,Machine Learning
4,"Psychotherapist, dance movement",Barker Group,Sydney,Entry Level,112000.0,Healthcare,"Nursing, Medical Research, Pharmaceuticals"


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Job Title         50000 non-null  object 
 1   Company           50000 non-null  object 
 2   Location          50000 non-null  object 
 3   Experience Level  50000 non-null  object 
 4   Salary            50000 non-null  float64
 5   Industry          50000 non-null  object 
 6   Required Skills   50000 non-null  object 
dtypes: float64(1), object(6)
memory usage: 2.7+ MB


In [10]:
df.isnull().sum()

,0
Job Title,0
Company,0
Location,0
Experience Level,0
Salary,0
Industry,0
Required Skills,0


In [11]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [12]:
print("Experience Levels:")
print(df["Experience Level"].unique())

print("\nIndustries:")
print(df["Industry"].unique())

print("\nLocations:")
print(df["Location"].unique())

Experience Levels:
['Senior Level' 'Mid Level' 'Entry Level']

Industries:
['Healthcare' 'Marketing' 'Software' 'Finance' 'Manufacturing' 'Retail'
 'Education']

Locations:
['Sydney' 'San Francisco' 'New York' 'Berlin' 'London' 'Bangalore'
 'Toronto']


In [13]:
df["Required Skills"].head(20)

,Required Skills
0,Pharmaceuticals
1,"Google Ads, SEO, Content Writing"
2,"Patient Care, Nursing, Medical Research, Pharm..."
3,Machine Learning
4,"Nursing, Medical Research, Pharmaceuticals"
5,"Financial Modeling, Risk Analysis"
6,Pharmaceuticals
7,"Java, Python"
8,"Python, SQL, Risk Analysis"
9,Production Planning


In [14]:
df_model = df.copy()

df_model["Required Skills"] = (
    df_model["Required Skills"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Model dataset created!")
print("Rows:", len(df_model))

Model dataset created!
Rows: 50000


In [15]:
df_model["Skill_List"] = df_model["Required Skills"].apply(
    lambda x: [
        skill.strip().lower()
        for skill in x.split(",")
        if skill.strip()
    ]
)

df_model[["Required Skills", "Skill_List"]].head(10)

,Required Skills,Skill_List
0,Pharmaceuticals,[pharmaceuticals]
1,"Google Ads, SEO, Content Writing","[google ads, seo, content writing]"
2,"Patient Care, Nursing, Medical Research, Pharm...","[patient care, nursing, medical research, phar..."
3,Machine Learning,[machine learning]
4,"Nursing, Medical Research, Pharmaceuticals","[nursing, medical research, pharmaceuticals]"
5,"Financial Modeling, Risk Analysis","[financial modeling, risk analysis]"
6,Pharmaceuticals,[pharmaceuticals]
7,"Java, Python","[java, python]"
8,"Python, SQL, Risk Analysis","[python, sql, risk analysis]"
9,Production Planning,[production planning]


In [16]:
user_profile = {
    "skills": ["python", "sql", "machine learning"],
    "experience": "Entry Level",
    "industry": "Software",
    "location": "Bangalore"
}

print("User profile:")
print(user_profile)

User profile:
{'skills': ['python', 'sql', 'machine learning'], 'experience': 'Entry Level', 'industry': 'Software', 'location': 'Bangalore'}


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Convert each job's skill list back into a text string
job_skill_text = df_model["Skill_List"].apply(lambda skills: " ".join(skills))

# Create TF-IDF model
vectorizer = TfidfVectorizer()

# Learn skill patterns from all jobs
job_skill_matrix = vectorizer.fit_transform(job_skill_text)

print("TF-IDF model created!")
print("Job-skill matrix shape:", job_skill_matrix.shape)

TF-IDF model created!
Job-skill matrix shape: (50000, 40)


In [18]:
# Convert user's skills into one text string
user_skill_text = " ".join(user_profile["skills"])

# Convert user skills into the same TF-IDF space as the jobs
user_skill_vector = vectorizer.transform([user_skill_text])

# Calculate similarity between user and every job
skill_similarity = cosine_similarity(
    user_skill_vector,
    job_skill_matrix
).flatten()

# Store the similarity score
df_model["Skill Similarity"] = skill_similarity

print("Skill similarity calculated!")
print(df_model["Skill Similarity"].head(10))

Skill similarity calculated!
0    0.000000
1    0.000000
2    0.000000
3    0.788586
4    0.000000
5    0.000000
6    0.000000
7    0.267419
8    0.397893
9    0.000000
Name: Skill Similarity, dtype: float64


In [19]:
df_model["Industry Match"] = (
    df_model["Industry"].str.lower()
    == user_profile["industry"].lower()
).astype(float)

print(df_model["Industry Match"].value_counts())

Industry Match
0.0    42698
1.0     7302
Name: count, dtype: int64


In [20]:
df_model["Experience Match"] = (
    df_model["Experience Level"].str.lower()
    == user_profile["experience"].lower()
).astype(float)

print(df_model["Experience Match"].value_counts())

Experience Match
0.0    33397
1.0    16603
Name: count, dtype: int64


In [21]:
df_model["Location Match"] = (
    df_model["Location"].str.lower()
    == user_profile["location"].lower()
).astype(float)

print(df_model["Location Match"].value_counts())

Location Match
0.0    42948
1.0     7052
Name: count, dtype: int64


In [22]:
user_profile["min_salary"] = 80000

df_model["Salary Score"] = (
    df_model["Salary"] / user_profile["min_salary"]
).clip(upper=1)

print(df_model["Salary Score"].head(10))

0    1.0000
1    0.6250
2    0.9625
3    1.0000
4    1.0000
5    1.0000
6    1.0000
7    1.0000
8    1.0000
9    0.8375
Name: Salary Score, dtype: float64


In [23]:
df_model["Final Score"] = (
    0.50 * df_model["Skill Similarity"]
    + 0.20 * df_model["Industry Match"]
    + 0.15 * df_model["Experience Match"]
    + 0.10 * df_model["Location Match"]
    + 0.05 * df_model["Salary Score"]
)

print("Final recommendation scores calculated!")
print(df_model["Final Score"].describe())

Final recommendation scores calculated!
count    50000.000000
mean         0.182931
std          0.172952
min          0.025000
25%          0.050000
50%          0.150000
75%          0.207801
max          1.000000
Name: Final Score, dtype: float64


In [24]:
top10 = df_model.sort_values(
    by="Final Score",
    ascending=False
).head(10)

top10[
    [
        "Job Title",
        "Company",
        "Location",
        "Experience Level",
        "Salary",
        "Industry",
        "Required Skills",
        "Final Score"
    ]
]

,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills,Final Score
43533,"Horticulturist, commercial",White-Cain,Bangalore,Entry Level,148000.0,Software,"SQL, C++, Python, Machine Learning",1.000000
26626,"Producer, television/film/video",Duran-Jones,Bangalore,Entry Level,136000.0,Software,"Python, SQL, Machine Learning, C++",1.000000
29109,"Lecturer, further education",Ayala-Bryant,Bangalore,Entry Level,87000.0,Software,"C++, Python, SQL, Machine Learning",1.000000
33590,Industrial/product designer,Maldonado Group,Bangalore,Entry Level,138000.0,Software,"Machine Learning, C++, SQL, Python",1.000000
14904,Research scientist (maths),Williams-Hill,Bangalore,Entry Level,53000.0,Software,"SQL, Machine Learning, Python",0.983125
43782,"Programmer, applications",Miller Group,Bangalore,Entry Level,50000.0,Software,"Machine Learning, Python, SQL",0.981250
25640,Equities trader,"Bell, Brown and Bryant",Bangalore,Entry Level,91000.0,Software,"SQL, Machine Learning",0.950450
35591,Armed forces operational officer,"Graham, Silva and Acosta",Bangalore,Entry Level,133000.0,Software,"Python, Machine Learning",0.950069
12765,"Engineer, mining","Richards, Butler and Reyes",Bangalore,Entry Level,75000.0,Software,"Python, Machine Learning",0.946944
13419,Adult guidance worker,Harper Group,Bangalore,Entry Level,73000.0,Software,"SQL, Machine Learning",0.946075


In [25]:
def get_skill_match(user_skills, job_skills):
    user_skills = set(skill.lower().strip() for skill in user_skills)
    job_skills = set(skill.lower().strip() for skill in job_skills)

    matched = user_skills.intersection(job_skills)
    missing = job_skills - user_skills

    return list(matched), list(missing)

In [26]:
recommended_job = top10.iloc[0]

matched, missing = get_skill_match(
    user_profile["skills"],
    recommended_job["Skill_List"]
)

print("Job:", recommended_job["Job Title"])
print("Matched Skills:", matched)
print("Missing Skills:", missing)

Job: Horticulturist, commercial
Matched Skills: ['machine learning', 'python', 'sql']
Missing Skills: ['c++']


In [27]:
def explain_recommendation(job, user_profile):
    matched, missing = get_skill_match(
        user_profile["skills"],
        job["Skill_List"]
    )

    reasons = []

    if job["Industry"].lower() == user_profile["industry"].lower():
        reasons.append("Industry matches your preference")

    if job["Experience Level"].lower() == user_profile["experience"].lower():
        reasons.append("Experience level matches")

    if job["Location"].lower() == user_profile["location"].lower():
        reasons.append("Location matches your preference")

    if job["Salary"] >= user_profile["min_salary"]:
        reasons.append("Meets your salary preference")

    return matched, missing, reasons

In [28]:
matched, missing, reasons = explain_recommendation(
    recommended_job,
    user_profile
)

print("JOB:", recommended_job["Job Title"])
print()
print("Why this job?")
for reason in reasons:
    print("✓", reason)

print()
print("Matched Skills:", matched)
print("Skills to Develop:", missing)

JOB: Horticulturist, commercial

Why this job?
✓ Industry matches your preference
✓ Experience level matches
✓ Location matches your preference
✓ Meets your salary preference

Matched Skills: ['machine learning', 'python', 'sql']
Skills to Develop: ['c++']


In [29]:
recommendation_report = []

for _, job in top10.iterrows():

    matched, missing, reasons = explain_recommendation(
        job,
        user_profile
    )

    recommendation_report.append({
        "Job Title": job["Job Title"],
        "Company": job["Company"],
        "Location": job["Location"],
        "Experience": job["Experience Level"],
        "Salary": job["Salary"],
        "Industry": job["Industry"],
        "Skill Similarity": round(job["Skill Similarity"], 3),
        "Final Score": round(job["Final Score"], 3),
        "Matched Skills": ", ".join(matched),
        "Skills to Develop": ", ".join(missing)
    })

recommendation_report = pd.DataFrame(recommendation_report)

recommendation_report

,Job Title,Company,Location,Experience,Salary,Industry,Skill Similarity,Final Score,Matched Skills,Skills to Develop
0,"Horticulturist, commercial",White-Cain,Bangalore,Entry Level,148000.0,Software,1.000,1.000,"machine learning, python, sql",c++
1,"Producer, television/film/video",Duran-Jones,Bangalore,Entry Level,136000.0,Software,1.000,1.000,"machine learning, python, sql",c++
2,"Lecturer, further education",Ayala-Bryant,Bangalore,Entry Level,87000.0,Software,1.000,1.000,"machine learning, python, sql",c++
3,Industrial/product designer,Maldonado Group,Bangalore,Entry Level,138000.0,Software,1.000,1.000,"machine learning, python, sql",c++
4,Research scientist (maths),Williams-Hill,Bangalore,Entry Level,53000.0,Software,1.000,0.983,"machine learning, sql, python",
5,"Programmer, applications",Miller Group,Bangalore,Entry Level,50000.0,Software,1.000,0.981,"machine learning, python, sql",
6,Equities trader,"Bell, Brown and Bryant",Bangalore,Entry Level,91000.0,Software,0.901,0.950,"machine learning, sql",
7,Armed forces operational officer,"Graham, Silva and Acosta",Bangalore,Entry Level,133000.0,Software,0.900,0.950,"machine learning, python",
8,"Engineer, mining","Richards, Butler and Reyes",Bangalore,Entry Level,75000.0,Software,0.900,0.947,"machine learning, python",
9,Adult guidance worker,Harper Group,Bangalore,Entry Level,73000.0,Software,0.901,0.946,"machine learning, sql",


In [30]:
print("Top 10 Recommendations")
print("=" * 80)

for i, (_, job) in enumerate(top10.iterrows(), start=1):
    print(f"{i}. {job['Job Title']}")
    print(f"   Company: {job['Company']}")
    print(f"   Location: {job['Location']}")
    print(f"   Experience: {job['Experience Level']}")
    print(f"   Industry: {job['Industry']}")
    print(f"   Skill Similarity: {job['Skill Similarity']:.3f}")
    print(f"   Final Score: {job['Final Score']:.3f}")
    print()

Top 10 Recommendations
1. Horticulturist, commercial
   Company: White-Cain
   Location: Bangalore
   Experience: Entry Level
   Industry: Software
   Skill Similarity: 1.000
   Final Score: 1.000

2. Producer, television/film/video
   Company: Duran-Jones
   Location: Bangalore
   Experience: Entry Level
   Industry: Software
   Skill Similarity: 1.000
   Final Score: 1.000

3. Lecturer, further education
   Company: Ayala-Bryant
   Location: Bangalore
   Experience: Entry Level
   Industry: Software
   Skill Similarity: 1.000
   Final Score: 1.000

4. Industrial/product designer
   Company: Maldonado Group
   Location: Bangalore
   Experience: Entry Level
   Industry: Software
   Skill Similarity: 1.000
   Final Score: 1.000

5. Research scientist (maths)
   Company: Williams-Hill
   Location: Bangalore
   Experience: Entry Level
   Industry: Software
   Skill Similarity: 1.000
   Final Score: 0.983

6. Programmer, applications
   Company: Miller Group
   Location: Bangalore
   Exper

In [31]:
def recommend_jobs(
    skills,
    experience,
    industry,
    location,
    min_salary,
    top_n=10
):

    # Create user profile
    user_skills = [
        skill.strip().lower()
        for skill in skills
        if skill.strip()
    ]

    user_skill_text = " ".join(user_skills)

    # Convert user skills into TF-IDF vector
    user_vector = vectorizer.transform([user_skill_text])

    # Calculate skill similarity with all jobs
    skill_scores = cosine_similarity(
        user_vector,
        job_skill_matrix
    ).flatten()

    # Create a copy so original data stays unchanged
    results = df_model.copy()

    results["Skill Similarity"] = skill_scores

    # Industry match
    results["Industry Match"] = (
        results["Industry"].str.lower()
        == industry.lower()
    ).astype(float)

    # Experience match
    results["Experience Match"] = (
        results["Experience Level"].str.lower()
        == experience.lower()
    ).astype(float)

    # Location match
    results["Location Match"] = (
        results["Location"].str.lower()
        == location.lower()
    ).astype(float)

    # Salary score
    results["Salary Score"] = (
        results["Salary"] / min_salary
    ).clip(upper=1)

    # Final recommendation score
    results["Final Score"] = (
        0.50 * results["Skill Similarity"]
        + 0.20 * results["Industry Match"]
        + 0.15 * results["Experience Match"]
        + 0.10 * results["Location Match"]
        + 0.05 * results["Salary Score"]
    )

    # Rank jobs
    results = results.sort_values(
        by="Final Score",
        ascending=False
    ).head(top_n)

    # Add explanations
    recommendations = []

    for _, job in results.iterrows():

        matched, missing = get_skill_match(
            user_skills,
            job["Skill_List"]
        )

        recommendations.append({
            "Job Title": job["Job Title"],
            "Company": job["Company"],
            "Location": job["Location"],
            "Experience": job["Experience Level"],
            "Salary": job["Salary"],
            "Industry": job["Industry"],
            "Final Score": round(job["Final Score"], 3),
            "Matched Skills": ", ".join(matched),
            "Skills to Develop": ", ".join(missing)
        })

    return pd.DataFrame(recommendations)

In [32]:
recommendations = recommend_jobs(
    skills=["python", "sql", "machine learning"],
    experience="Entry Level",
    industry="Software",
    location="Bangalore",
    min_salary=80000,
    top_n=10
)

recommendations

,Job Title,Company,Location,Experience,Salary,Industry,Final Score,Matched Skills,Skills to Develop
0,"Horticulturist, commercial",White-Cain,Bangalore,Entry Level,148000.0,Software,1.000,"machine learning, python, sql",c++
1,"Producer, television/film/video",Duran-Jones,Bangalore,Entry Level,136000.0,Software,1.000,"machine learning, python, sql",c++
2,"Lecturer, further education",Ayala-Bryant,Bangalore,Entry Level,87000.0,Software,1.000,"machine learning, python, sql",c++
3,Industrial/product designer,Maldonado Group,Bangalore,Entry Level,138000.0,Software,1.000,"machine learning, python, sql",c++
4,Research scientist (maths),Williams-Hill,Bangalore,Entry Level,53000.0,Software,0.983,"machine learning, sql, python",
5,"Programmer, applications",Miller Group,Bangalore,Entry Level,50000.0,Software,0.981,"machine learning, python, sql",
6,Equities trader,"Bell, Brown and Bryant",Bangalore,Entry Level,91000.0,Software,0.950,"machine learning, sql",
7,Armed forces operational officer,"Graham, Silva and Acosta",Bangalore,Entry Level,133000.0,Software,0.950,"machine learning, python",
8,"Engineer, mining","Richards, Butler and Reyes",Bangalore,Entry Level,75000.0,Software,0.947,"machine learning, python",
9,Adult guidance worker,Harper Group,Bangalore,Entry Level,73000.0,Software,0.946,"machine learning, sql",


In [33]:
test_recommendations = recommend_jobs(
    skills=["java", "python", "c++"],
    experience="Entry Level",
    industry="Software",
    location="London",
    min_salary=70000,
    top_n=5
)

test_recommendations

,Job Title,Company,Location,Experience,Salary,Industry,Final Score,Matched Skills,Skills to Develop
0,Air broker,Buckley-Huffman,London,Entry Level,110000.0,Software,1.000,"java, python",
1,Systems developer,"Melton, Olson and Jackson",London,Entry Level,80000.0,Software,1.000,"java, python",
2,Publishing rights manager,Miller Group,London,Entry Level,135000.0,Software,1.000,"java, python",
3,Illustrator,Watts Group,London,Entry Level,126000.0,Software,0.925,"java, python, c++",sql
4,"Administrator, local government",Cortez-Martinez,London,Entry Level,148000.0,Software,0.925,"java, python",sql


In [34]:
import pickle

with open("job_data.pkl", "wb") as file:
    pickle.dump(df_model, file)

print("job_data.pkl saved successfully!")

job_data.pkl saved successfully!


In [35]:
with open("tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

print("tfidf_vectorizer.pkl saved successfully!")

tfidf_vectorizer.pkl saved successfully!


In [36]:
from scipy.sparse import save_npz

save_npz("job_skill_matrix.npz", job_skill_matrix)

print("job_skill_matrix.npz saved successfully!")

job_skill_matrix.npz saved successfully!
